# F3 homework: PCA from scratch

In this notebook you will implement principal component analysis (PCA) with nothing but NumPy, check it
against scikit-learn, and use it to draw a 4-dimensional dataset in 2-D.

The dataset is the classic **iris** flowers: 150 flowers from 3 species, each described by 4
measurements in centimetres (sepal length, sepal width, petal length, petal width). Four numbers per
flower means each flower is a point in $\mathbb{R}^4$, which nobody can draw. PCA finds the 2-D view
that keeps as much of the spread as possible.

**How to work.** Each task has a `TODO` cell, followed by a check cell that prints `OK` when your code
is right. Run the cells in order. If a check fails, read its message, fix your function and run both
cells again. Everything runs in seconds on any laptop or on Colab; there is no GPU work here.

You will use ideas from these F3 lessons: "Dot product, length and cosine similarity",
"Matrix multiplication, identity and transpose", "Eigenvectors and eigenvalues" and "SVD and PCA".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data              # shape (150, 4)
y = iris.target            # 0, 1, 2: the species
feature_names = iris.feature_names
species = iris.target_names
print(X.shape, feature_names)
print(X[:3])

## Task 1: standardise the features

PCA chases variance, and variance depends on units. The four iris measurements are all in centimetres,
but petal length varies far more than sepal width, so we standardise: subtract each column's mean and
divide by each column's standard deviation. After that every feature has mean 0 and standard
deviation 1.

Write `standardise(X)` that returns the standardised copy. Use `X.mean(axis=0)` and `X.std(axis=0)`
(the population standard deviation, which is what scikit-learn's `StandardScaler` uses).

In [ ]:
def standardise(X):
    """Return X with every column shifted to mean 0 and scaled to standard deviation 1."""
    # TODO
    raise NotImplementedError

Xs = standardise(X)

In [ ]:
from sklearn.preprocessing import StandardScaler
assert Xs.shape == (150, 4), Xs.shape
assert np.allclose(Xs.mean(axis=0), 0), "every column should have mean 0"
assert np.allclose(Xs.std(axis=0), 1), "every column should have standard deviation 1"
assert np.allclose(Xs, StandardScaler().fit_transform(X)), "should match StandardScaler"
assert np.allclose(standardise(np.array([[1.0, 10.0], [3.0, 30.0]])), [[-1, -1], [1, 1]])
print("OK")

## Task 2: the covariance matrix

For centred data $X_c$ with $N$ rows, the covariance matrix is

$$C = \frac{1}{N - 1} X_c^\top X_c$$

Entry $C_{jk}$ measures how features $j$ and $k$ vary together; the diagonal holds each feature's
variance. Write `covariance(X)` that centres `X` (subtract the column means; it does no harm if `X` is
already centred) and returns $C$. Use `@` and `.T`, not `np.cov` (the check uses `np.cov`).

In [ ]:
def covariance(X):
    """Return the (d, d) covariance matrix of the (N, d) data matrix X."""
    # TODO: centre, then Xc.T @ Xc / (N - 1)
    raise NotImplementedError

C = covariance(Xs)
print(np.round(C, 3))

In [ ]:
assert C.shape == (4, 4), C.shape
assert np.allclose(C, C.T), "a covariance matrix is symmetric"
assert np.allclose(C, np.cov(Xs, rowvar=False)), "should match np.cov(Xs, rowvar=False)"
assert np.allclose(covariance(X), np.cov(X, rowvar=False)), "remember to centre inside covariance()"
print("OK")

## Task 3: PCA with eigenvectors

The principal components are the eigenvectors of $C$, sorted by eigenvalue from largest to smallest,
and each eigenvalue is the variance along its component.

Write `pca_eig(X, k)` that returns `(components, variances)`:

- `components`: shape `(k, d)`, one unit-length component per **row**, largest variance first;
- `variances`: shape `(k,)`, the matching eigenvalues.

Use `np.linalg.eigh` (the matrix is symmetric). Remember that `eigh` returns eigenvalues in
**ascending** order and eigenvectors as **columns**. `np.argsort(vals)[::-1]` gives the indices from
largest to smallest.

In [ ]:
def pca_eig(X, k):
    """Return the top-k principal components (rows) and their variances, from the covariance eigenvectors."""
    # TODO
    raise NotImplementedError

comps_eig, var_eig = pca_eig(Xs, 4)
print(np.round(var_eig, 4))
print(np.round(comps_eig, 3))

In [ ]:
c2, v2 = pca_eig(Xs, 2)
assert c2.shape == (2, 4) and v2.shape == (2,), (c2.shape, v2.shape)
assert np.all(np.diff(var_eig) <= 0), "variances should be sorted from largest to smallest"
assert np.allclose(np.linalg.norm(comps_eig, axis=1), 1), "every component should have length 1"
for v, lam in zip(comps_eig, var_eig):
    assert np.allclose(C @ v, lam * v), "each row should be an eigenvector of C with its variance as eigenvalue"
assert np.allclose(comps_eig @ comps_eig.T, np.eye(4)), "the components should be mutually perpendicular"
print("OK")

## Task 4: PCA with the SVD

The SVD route skips the covariance matrix. Centre the data, take
`U, s, Vt = np.linalg.svd(Xc, full_matrices=False)`; then the rows of `Vt` are the components and
`s**2 / (N - 1)` are the variances.

Write `pca_svd(X, k)` that returns `(components, variances, Z)` where `Z = Xc @ components.T` is the
data projected onto the top `k` components, shape `(N, k)`.

In [ ]:
def pca_svd(X, k):
    """Return the top-k components (rows), their variances and the projected data Z, using the SVD."""
    # TODO
    raise NotImplementedError

comps, variances, Z = pca_svd(Xs, 2)
print(np.round(variances, 4))
print(Z[:3])

In [ ]:
assert comps.shape == (2, 4) and variances.shape == (2,) and Z.shape == (150, 2), (comps.shape, variances.shape, Z.shape)
assert np.allclose(variances, var_eig[:2]), "the SVD variances should equal the eigenvalues from Task 3"
signs_eig = np.sign(np.sum(comps * comps_eig[:2], axis=1))
assert np.allclose(comps * signs_eig[:, None], comps_eig[:2]), "each component should match Task 3, or be its negative (a whole row flips sign)"
assert np.allclose(Z.var(axis=0, ddof=1), variances), "the variance of each column of Z should be that component's variance"
print("OK")

## Task 5: explained variance

Write `explained_variance_ratio(X)` that returns, for **all** $d$ components, the fraction of the
total variance each one explains (an array of shape `(d,)` that sums to 1). You may reuse `pca_svd`.

Then make a bar chart of the ratios with a line for the running total (`np.cumsum`). How many
components do you need to keep 95% of the variance?

In [ ]:
def explained_variance_ratio(X):
    """Return the fraction of total variance explained by each principal component, largest first."""
    # TODO
    raise NotImplementedError

ratio = explained_variance_ratio(Xs)
print(np.round(ratio, 4), np.round(np.cumsum(ratio), 4))

# TODO: bar chart of ratio and a line of np.cumsum(ratio)


In [ ]:
from sklearn.decomposition import PCA
assert ratio.shape == (4,), ratio.shape
assert np.isclose(ratio.sum(), 1), "the ratios should sum to 1"
assert np.allclose(ratio, PCA().fit(Xs).explained_variance_ratio_), "should match sklearn's explained_variance_ratio_"
print("OK")

## Task 6: the 2-D picture

Scatter-plot the projected data `Z` from Task 4: first principal component on the x-axis, second on
the y-axis, one colour per species (loop over `0, 1, 2` and select rows with `y == i`), with a legend
using `species[i]`. Label the axes with the percentage of variance each component explains.

Look at the plot. Which species separates cleanly from the others? Write one or two sentences in the
markdown cell below.

In [ ]:
# TODO: scatter plot of Z coloured by species


**Your observation:** _(which species separates, and along which component?)_

## Task 7: compare with scikit-learn

Fit `PCA(n_components=2)` from scikit-learn to `Xs` and compare:

- `pca.components_` with your `comps`,
- `pca.explained_variance_` with your `variances`,
- `pca.transform(Xs)` with your `Z`.

Components are only defined **up to sign**: a whole row may be flipped, so align the signs row by row before comparing. Store the fitted model as
`sk_pca` and its transform as `Z_sk`.

In [ ]:
from sklearn.decomposition import PCA

# TODO: fit sklearn's PCA and compute Z_sk
sk_pca = None
Z_sk = None

In [ ]:
assert sk_pca is not None and Z_sk is not None, "fit sk_pca and compute Z_sk"
signs = np.sign(np.sum(sk_pca.components_ * comps, axis=1))
assert np.allclose(comps * signs[:, None], sk_pca.components_), "each component should match sklearn's, or be its negative (a whole row flips sign)"
assert np.allclose(sk_pca.explained_variance_, variances), "variances should match"
assert np.allclose(Z_sk, Z * signs), "after flipping signs where needed, the projections are identical"
print("OK")

## Task 8: reconstruction error

Projecting to $k$ dimensions throws information away. To see how much, map back:
$\hat{X}_c = Z_k V_k^\top$ (with $Z_k$ of shape `(N, k)` and the components $V_k^\top$ of shape `(k, d)`),
then measure the mean squared error against the centred data.

Write `reconstruction_error(X, k)` that returns `np.mean((Xc - Xc_hat) ** 2)`. Print it for
`k = 1, 2, 3, 4`. It should shrink as `k` grows and be (almost) 0 at `k = 4`.

In [ ]:
def reconstruction_error(X, k):
    """Return the mean squared error after projecting centred X onto k components and mapping back."""
    # TODO
    raise NotImplementedError

errors = [reconstruction_error(Xs, k) for k in range(1, 5)]
print(np.round(errors, 4))

In [ ]:
assert all(a > b for a, b in zip(errors, errors[1:])), "the error should shrink as k grows"
assert errors[-1] < 1e-20, "with all 4 components the reconstruction should be exact"
N = Xs.shape[0]
dropped = np.cumsum(ratio[::-1])[::-1]  # fraction of variance in components k+1..d
for k in range(1, 4):
    expected = dropped[k] * np.trace(C) * (N - 1) / (N * 4)
    assert np.isclose(errors[k - 1], expected), f"k={k}: the error should equal the variance of the dropped components (scaled)"
print("OK")

## Stretch tasks (optional)

1. Run PCA on the **raw** `X` instead of `Xs`. Print the first component and the explained variance
   ratio. Which feature dominates, and why? (Hint: compare the variances of the raw columns.)
2. Load `sklearn.datasets.load_digits()` (8×8 images, so 64 features), run your `pca_svd` with `k = 2`
   and scatter-plot the digits coloured by their label. How many components keep 90% of the variance?
3. Write `pca_power(X, k)` that finds the top component with power iteration (from the eigenvectors
   lesson) instead of `eigh` or `svd`, and compare it with Task 3.